# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import duckdb, getpass
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# Check dim_content's real schema first — don't guess column names again (lesson from w03_data_contract)
dim_content_schema = conn.execute(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet')").df()
print(dim_content_schema.to_string())

Paste your Hugging Face READ token (hf_...): ··········
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    Non

In [3]:
grain_check = conn.execute(f"""
    SELECT content_hash_id, COUNT(*) AS c
    FROM read_parquet('{REL}/dim_content.parquet')
    GROUP BY content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print(grain_check)
print(f"\nContent items with duplicates: {len(grain_check)}")

total_rows = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{REL}/dim_content.parquet')").fetchone()[0]
distinct_content = conn.execute(f"SELECT COUNT(DISTINCT content_hash_id) FROM read_parquet('{REL}/dim_content.parquet')").fetchone()[0]
print(f"Total rows: {total_rows:,} | Distinct content_hash_id: {distinct_content:,}")

Empty DataFrame
Columns: [content_hash_id, c]
Index: []

Content items with duplicates: 0
Total rows: 519,606 | Distinct content_hash_id: 519,606


In [4]:
feature_query = f"""
    WITH daily AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions) AS sum_gsc_impressions_mar,
            AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_gsc_position_mar,
            SUM(scroll_events) FILTER (WHERE ga4_data_available IS TRUE) AS sum_scroll_events_mar,
            MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_mar,
            SUM(gsc_clicks) AS sum_gsc_clicks_mar,
            (SUM(gsc_clicks) > 0)::INTEGER AS has_search_clicks_label
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT
        d.*,
        c.content_type,
        c.word_count,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
    FROM daily d
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
      ON d.content_hash_id = c.content_hash_id
    WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
"""
features_df = conn.execute(feature_query).df()
print(features_df.shape)
print(features_df[["content_age_days"]].describe())
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176568, 11)
       content_age_days
count     176568.000000
mean         184.721750
std          123.646347
min            0.000000
25%           69.000000
50%          193.000000
75%          260.000000
max          494.000000


,client_hash_id,content_hash_id,sum_gsc_impressions_mar,avg_gsc_position_mar,sum_scroll_events_mar,ga4_available_mar,sum_gsc_clicks_mar,has_search_clicks_label,content_type,word_count,content_age_days
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,4.394234,NaN,0,2.0,1,keyword article,<NA>,396
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,7.842593,NaN,0,0.0,0,keyword article,<NA>,396
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,8.454069,0.0,1,0.0,0,keyword article,<NA>,396
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.320337,1.0,1,6.0,1,keyword article,<NA>,396
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,4.459107,0.0,1,16.0,1,keyword article,2475,396


In [5]:
features_df["content_type"] = features_df["content_type"].fillna("unknown")
content_type_dummies = pd.get_dummies(features_df["content_type"], prefix="content_type")

features_df["word_count"] = features_df["word_count"].fillna(features_df["word_count"].median())
features_df["content_age_days"] = features_df["content_age_days"].fillna(features_df["content_age_days"].median())
features_df["sum_scroll_events_mar"] = features_df["sum_scroll_events_mar"].fillna(0)  # honest: 0 only meaningful alongside ga4_available_mar

model_df = pd.concat([features_df, content_type_dummies], axis=1)
print(f"Content types found: {features_df['content_type'].unique()}")

Content types found: ['keyword article' 'feedly article' 'comparison article']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
feature_notes = pd.DataFrame([
    {"feature": "sum_gsc_impressions_mar", "meaning": "total search impressions in March", "available_when": "Known by end of decision month — a real-time count as impressions accrue."},
    {"feature": "avg_gsc_position_mar", "meaning": "mean search ranking position in March", "available_when": "Same — updates daily from GSC, available before any future-window label."},
    {"feature": "sum_scroll_events_mar + ga4_available_mar", "meaning": "GA4 engagement volume, with an explicit availability flag", "available_when": "Available only for clients with GA4 wired up; flag prevents silently treating 'no GA4' as 'zero engagement'."},
    {"feature": "content_type (one-hot)", "meaning": "article format category", "available_when": "Fixed at publish time — always knowable before any prediction window."},
    {"feature": "word_count, content_age_days", "meaning": "static content properties", "available_when": "Set at creation/last edit — knowable before the decision moment, doesn't depend on future performance."},
])
print(feature_notes.to_string(index=False))

                                  feature                                                   meaning                                                                                               available_when
                  sum_gsc_impressions_mar                         total search impressions in March                                    Known by end of decision month — a real-time count as impressions accrue.
                     avg_gsc_position_mar                     mean search ranking position in March                                     Same — updates daily from GSC, available before any future-window label.
sum_scroll_events_mar + ga4_available_mar GA4 engagement volume, with an explicit availability flag Available only for clients with GA4 wired up; flag prevents silently treating 'no GA4' as 'zero engagement'.
                   content_type (one-hot)                                   article format category                                        Fixed at publish time — a

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
honest_cols = ["sum_gsc_impressions_mar", "avg_gsc_position_mar", "sum_scroll_events_mar",
               "ga4_available_mar", "word_count", "content_age_days"] + list(content_type_dummies.columns)

X = model_df[honest_cols].fillna(0)
y = model_df["has_search_clicks_label"]

clients = model_df["client_hash_id"].unique()
np.random.seed(42)
np.random.shuffle(clients)
test_clients = set(clients[:max(1, int(len(clients) * 0.2))])
test_mask = model_df["client_hash_id"].isin(test_clients)

X_train, X_test = X[~test_mask], X[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

honest_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"HONEST AUC: {honest_auc:.4f}")

# Now spring the trap: add the label-derived column on purpose
X_leak = X.copy()
X_leak["sum_gsc_clicks_mar"] = model_df["sum_gsc_clicks_mar"].fillna(0)
X_leak_train, X_leak_test = X_leak[~test_mask], X_leak[test_mask]

leak_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_leak_train, y_train)
leak_auc = roc_auc_score(y_test, leak_model.predict_proba(X_leak_test)[:, 1])
print(f"LEAKED AUC (with sum_gsc_clicks_mar added): {leak_auc:.4f}")
print(f"Jump: {leak_auc - honest_auc:.4f} — this is the model reading the answer key, not learning.")

# Delete the leak, keep only the honest number
final_auc = honest_auc
print(f"\nFinal, trusted AUC: {final_auc:.4f} (leaked version discarded)")

HONEST AUC: 0.8883
LEAKED AUC (with sum_gsc_clicks_mar added): 1.0000
Jump: 0.1117 — this is the model reading the answer key, not learning.

Final, trusted AUC: 0.8883 (leaked version discarded)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [8]:
excluded = pd.DataFrame([
    {"field": "sum_gsc_clicks_mar", "why": "Exact source of the label — leakage, demonstrated in section 3."},
    {"field": "trend_direction / trend_pct", "why": "Computed from the future/label window — never a feature."},
    {"field": "content_hash_id / client_hash_id", "why": "Identifiers — for joins and client-holdout splitting only, never model inputs."},
    {"field": "health_score / priority_score", "why": "Product decision flags — not shipped in this data by design; would leak the decision itself if reconstructed."},
    {"field": "report_date", "why": "Context for windowing/splits — not a per-row predictive signal on its own."},
])
print(excluded.to_string(index=False))

                           field                                                                                                           why
              sum_gsc_clicks_mar                                               Exact source of the label — leakage, demonstrated in section 3.
     trend_direction / trend_pct                                                      Computed from the future/label window — never a feature.
content_hash_id / client_hash_id                                Identifiers — for joins and client-holdout splitting only, never model inputs.
   health_score / priority_score Product decision flags — not shipped in this data by design; would leak the decision itself if reconstructed.
                     report_date                                    Context for windowing/splits — not a per-row predictive signal on its own.


## Self-check

Before you submit, confirm each line honestly:

- [T] Every section above is filled — markdown thinking AND the code that backs it
- [T] The notebook runs top to bottom with no errors (Runtime → Run all)
- [T] No client names, URLs, or private queries anywhere
- [T] My claims use careful words: observed, measured, directional, decision-support
- [T] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.